<a href="https://colab.research.google.com/github/hongkyuh/NowBlur/blob/main/9%EC%A3%BC%EC%B0%A8_%EA%B0%95%EC%9D%98_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 사이킷런(sklearn)에서 제공하는 샘플 이미지 데이터를 불러오기 위한 함수 import
from sklearn.datasets import load_sample_images

# TensorFlow 라이브러리 import
# Keras의 전처리 레이어(CenterCrop, Rescaling)를 사용하기 위해 필요
import tensorflow as tf


# load_sample_images()는 사이킷런에 내장된 샘플 이미지 2장을 불러옵니다.
# 반환값은 딕셔너리 형태이며, 그중 "images" 키에 실제 이미지 배열이 들어 있습니다.
# images의 원본 형태는 보통 (2, 427, 640, 3)입니다.
# 의미:
# - 2: 이미지 개수
# - 427: 이미지 높이
# - 640: 이미지 너비
# - 3: RGB 색상 채널
images = load_sample_images()["images"]


# CenterCrop 레이어는 이미지의 중앙 부분을 잘라내는 전처리 층입니다.
# height=70, width=120은 각 이미지를 높이 70픽셀, 너비 120픽셀로 자르겠다는 뜻입니다.
# 이미지의 중심을 기준으로 잘라내므로 가장자리 부분은 제거됩니다.
# 입력 images의 형태가 (2, 427, 640, 3)이었다면,
# 처리 후 형태는 (2, 70, 120, 3)이 됩니다.
images = tf.keras.layers.CenterCrop(
    height=70,   # 잘라낼 이미지의 높이를 70픽셀로 지정
    width=120    # 잘라낼 이미지의 너비를 120픽셀로 지정
)(images)


# Rescaling 레이어는 픽셀값의 범위를 조정하는 전처리 층입니다.
# 일반적인 이미지 픽셀값은 0~255 범위의 정수입니다.
# scale=1 / 255를 적용하면 모든 픽셀값을 255로 나누어
# 0~1 범위의 실수값으로 변환합니다.
# 딥러닝 모델은 보통 0~1 범위로 정규화된 입력을 더 잘 처리합니다.
# 이미지의 크기(shape)는 바뀌지 않고, 픽셀값의 범위만 바뀝니다.
images = tf.keras.layers.Rescaling(
    scale=1 / 255   # 픽셀값을 255로 나누어 0~1 사이 값으로 변환
)(images)


# 전처리가 끝난 images 텐서의 크기(shape)를 확인합니다.
# 출력 결과:
# TensorShape([2, 70, 120, 3])
# 의미:
# - 2: 이미지 2장
# - 70: 각 이미지의 높이
# - 120: 각 이미지의 너비
# - 3: RGB 색상 채널
images.shape

In [ ]:
# Conv2D는 2D 합성곱 층을 만드는 Keras 레이어입니다.
# 주로 이미지 데이터에서 특징(feature)을 추출하는 데 사용됩니다.
# 입력 이미지는 일반적으로 (배치 크기, 높이, 너비, 채널 수) 형태를 가집니다.
conv_layer = tf.keras.layers.Conv2D(
    filters=32,      # 사용할 필터(커널)의 개수입니다.
                     # 필터가 32개이므로 출력 feature map도 32개 생성됩니다.
                     # 따라서 출력 텐서의 마지막 차원은 32가 됩니다.

    kernel_size=7    # 필터의 크기를 7x7로 지정합니다.
                     # 즉, 각 필터는 이미지의 7x7 영역을 보면서 특징을 추출합니다.
                     # kernel_size=7은 kernel_size=(7, 7)과 같은 의미입니다.
)


# 앞에서 CenterCrop과 Rescaling으로 전처리한 images 텐서를
# Conv2D 층에 입력합니다.
#
# images의 shape:
# (2, 70, 120, 3)
#
# 의미:
# - 2: 이미지 개수
# - 70: 이미지 높이
# - 120: 이미지 너비
# - 3: RGB 채널 수
#
# Conv2D 층은 각 이미지에 32개의 7x7 필터를 적용하여
# 32개의 feature map을 생성합니다.
fmaps = conv_layer(images)


# 합성곱 층을 통과한 결과 텐서의 크기를 확인합니다.
# 출력 결과:
# TensorShape([2, 64, 114, 32])
#
# 왜 높이와 너비가 줄어드는가?
# Conv2D의 기본 padding 값은 "valid"입니다.
# "valid" padding은 이미지 가장자리에 0을 추가하지 않고,
# 필터가 이미지 내부에서 완전히 겹치는 위치에만 합성곱을 수행합니다.
#
# 따라서 출력 높이와 너비는 다음처럼 계산됩니다.
#
# 출력 높이 = 입력 높이 - 커널 높이 + 1
#          = 70 - 7 + 1
#          = 64
#
# 출력 너비 = 입력 너비 - 커널 너비 + 1
#          = 120 - 7 + 1
#          = 114
#
# 출력 채널 수는 filters=32이므로 32입니다.
#
# 최종 출력 shape의 의미:
# - 2: 이미지 2장
# - 64: 합성곱 후 feature map의 높이
# - 114: 합성곱 후 feature map의 너비
# - 32: 생성된 feature map 개수
fmaps.shape

In [ ]:
# padding="same"을 사용한 Conv2D 층을 생성합니다.
# Conv2D는 이미지에서 특징(feature)을 추출하는 2D 합성곱 층입니다.
conv_layer = tf.keras.layers.Conv2D(
    filters=32,       # 필터(커널)의 개수입니다.
                      # 필터가 32개이므로 출력 feature map도 32개 생성됩니다.
                      # 따라서 출력 텐서의 마지막 차원은 32가 됩니다.

    kernel_size=7,    # 필터 크기를 7x7로 지정합니다.
                      # 각 필터는 이미지의 7x7 영역을 보면서 특징을 추출합니다.

    padding="same"    # 출력 feature map의 높이와 너비가 입력 이미지와 같아지도록
                      # 입력 이미지 가장자리에 0 패딩을 자동으로 추가합니다.
                      #
                      # 여기서 입력 images의 크기는 (2, 70, 120, 3)입니다.
                      # padding="same"을 사용하면 출력의 높이와 너비가
                      # 입력과 동일하게 70, 120으로 유지됩니다.
                      #
                      # 단, 채널 수는 filters=32에 의해 32로 바뀝니다.
)


# 전처리된 images 텐서를 합성곱 층에 입력합니다.
#
# 입력 images의 shape:
# (2, 70, 120, 3)
#
# 의미:
# - 2: 이미지 개수
# - 70: 이미지 높이
# - 120: 이미지 너비
# - 3: RGB 색상 채널
#
# padding="same"이므로 합성곱 후에도
# 높이와 너비는 70, 120으로 유지됩니다.
fmaps = conv_layer(images)


# 합성곱 층을 통과한 결과 텐서의 크기를 확인합니다.
#
# 출력 결과:
# TensorShape([2, 70, 120, 32])
#
# 의미:
# - 2: 이미지 2장
# - 70: 출력 feature map의 높이
# - 120: 출력 feature map의 너비
# - 32: 생성된 feature map 개수
#
# padding="valid"와 비교:
# - padding="valid"는 패딩을 추가하지 않으므로
#   출력 크기가 줄어듭니다.
#   예: (2, 70, 120, 3) → (2, 64, 114, 32)
#
# - padding="same"은 입력 가장자리에 0을 추가하여
#   출력 높이와 너비를 입력과 같게 유지합니다.
#   예: (2, 70, 120, 3) → (2, 70, 120, 32)
fmaps.shape

In [ ]:
# Conv2D 층도 Dense 층처럼 학습 가능한 가중치(weight)를 가지고 있습니다.
# Conv2D 층의 주요 가중치는 크게 두 가지입니다.
#
# 1. kernel:
#    이미지에서 특징을 추출하는 필터들의 가중치입니다.
#
# 2. bias:
#    각 필터가 만든 feature map에 더해지는 편향값입니다.
#
# 여기서는 앞에서 만든 conv_layer의 가중치를 확인합니다.
# 예:
# conv_layer = tf.keras.layers.Conv2D(
#     filters=32,
#     kernel_size=7,
#     padding="same"
# )
#
# 주의:
# Conv2D 층은 실제 입력 데이터를 한 번 통과해야 가중치가 생성됩니다.
# 따라서 아래 코드를 실행하기 전에 반드시 다음 코드가 먼저 실행되어 있어야 합니다.
#
# fmaps = conv_layer(images)


# get_weights()는 Conv2D 층이 가진 가중치를 넘파이 배열 형태로 반환합니다.
# Conv2D 층의 경우 보통 [커널, 편향] 순서로 반환됩니다.
kernels, biases = conv_layer.get_weights()


# kernels는 Conv2D 층의 커널, 즉 필터들의 가중치입니다.
#
# 출력 결과:
# (7, 7, 3, 32)
#
# 의미:
# - 7: 커널의 높이
# - 7: 커널의 너비
# - 3: 입력 채널 수
# - 32: 필터 개수
#
# 즉, 이 Conv2D 층에는 7x7 크기의 필터가 32개 있고,
# 각 필터는 RGB 이미지의 3개 채널을 모두 사용합니다.
#
# 하나의 필터 크기:
# 7 x 7 x 3
#
# 전체 커널 가중치 개수:
# 7 x 7 x 3 x 32 = 4,704개
kernels.shape


# biases는 Conv2D 층의 편향값입니다.
#
# 출력 결과:
# (32,)
#
# 의미:
# - 필터가 32개이므로 편향도 32개입니다.
# - 각 필터마다 하나의 bias 값이 있습니다.
#
# bias는 각 필터가 만든 feature map 전체에 더해지는 값입니다.
biases.shape

In [ ]:
# TensorFlow 라이브러리 import
# 사용자 정의 Keras 층을 만들기 위해 필요합니다.
import tensorflow as tf


# DepthPool은 사용자가 직접 정의한 Keras 층입니다.
# 이 층은 일반적인 MaxPooling2D처럼 높이와 너비 방향으로 풀링하는 것이 아니라,
# 채널 방향, 즉 depth 방향으로 최대 풀링을 수행합니다.
#
# 예를 들어 입력 텐서의 shape이 다음과 같다고 가정합니다.
#
# (배치 크기, 높이, 너비, 채널 수)
# (2, 70, 120, 32)
#
# pool_size=2라면 채널을 2개씩 묶어서 그중 가장 큰 값만 남깁니다.
# 따라서 채널 수가 32에서 16으로 줄어듭니다.
class DepthPool(tf.keras.layers.Layer):

    # __init__()은 층이 생성될 때 한 번 실행되는 초기화 메서드입니다.
    # pool_size는 채널을 몇 개씩 묶어서 최대값을 구할지 정하는 값입니다.
    def __init__(self, pool_size=2, **kwargs):

        # 부모 클래스(tf.keras.layers.Layer)의 초기화 메서드를 호출합니다.
        # name, dtype 같은 Keras 층 공통 옵션을 사용할 수 있게 해 줍니다.
        super().__init__(**kwargs)

        # 사용자가 지정한 pool_size 값을 객체 내부에 저장합니다.
        # 나중에 call() 메서드에서 사용됩니다.
        self.pool_size = pool_size

    # call()은 실제로 입력 데이터가 이 층을 통과할 때 실행되는 메서드입니다.
    # 즉, depth_pool(inputs)처럼 층을 호출하면 내부적으로 call(inputs)가 실행됩니다.
    def call(self, inputs):

        # 입력 텐서의 동적인 shape을 가져옵니다.
        # tf.shape(inputs)는 텐서 형태의 shape을 반환합니다.
        #
        # 예:
        # inputs.shape이 (2, 70, 120, 32)라면
        # shape은 [2, 70, 120, 32]가 됩니다.
        shape = tf.shape(inputs)

        # shape[-1]은 마지막 차원, 즉 채널 수를 의미합니다.
        #
        # 예:
        # shape = [2, 70, 120, 32]
        # shape[-1] = 32
        #
        # pool_size=2이면 채널을 2개씩 묶으므로
        # groups = 32 // 2 = 16
        #
        # groups는 채널 그룹의 개수입니다.
        groups = shape[-1] // self.pool_size

        # reshape을 위해 새로운 shape을 만듭니다.
        #
        # 기존 shape:
        # [배치 크기, 높이, 너비, 채널 수]
        #
        # 새 shape:
        # [배치 크기, 높이, 너비, groups, pool_size]
        #
        # 예:
        # 기존 shape = [2, 70, 120, 32]
        # pool_size = 2
        # groups = 16
        #
        # 새 shape = [2, 70, 120, 16, 2]
        #
        # 이렇게 바꾸면 마지막 채널 32개를
        # 2개씩 묶은 16개의 그룹으로 볼 수 있습니다.
        new_shape = tf.concat(
            [
                shape[:-1],                  # 마지막 차원인 채널 수를 제외한 부분
                [groups, self.pool_size]     # 채널을 groups개 그룹과 pool_size 크기로 분리
            ],
            axis=0
        )

        # 입력 텐서를 new_shape로 변형합니다.
        #
        # 예:
        # (2, 70, 120, 32)
        # → (2, 70, 120, 16, 2)
        #
        # 그 후 axis=-1 방향, 즉 마지막 차원(pool_size 방향)에서 최대값을 구합니다.
        #
        # 결과:
        # (2, 70, 120, 16, 2)
        # → (2, 70, 120, 16)
        #
        # 즉, 채널을 2개씩 묶어서 각 묶음의 최대값만 남기므로
        # 채널 수가 32에서 16으로 줄어듭니다.
        return tf.reduce_max(
            tf.reshape(inputs, new_shape),
            axis=-1
        )


# DepthPool 층 객체를 생성합니다.
# pool_size=2이므로 채널을 2개씩 묶어서 최대값을 계산합니다.
depth_pool = DepthPool(pool_size=2)


# 예시 입력으로 앞에서 만든 fmaps를 사용할 수 있습니다.
#
# fmaps의 shape이 다음과 같다면:
# TensorShape([2, 70, 120, 32])
#
# DepthPool을 적용한 출력은:
# TensorShape([2, 70, 120, 16])
#
# 높이와 너비는 그대로 유지되고,
# 채널 수만 32에서 16으로 줄어듭니다.
pooled_fmaps = depth_pool(fmaps)


# 깊이 방향 최대 풀링을 적용한 결과 텐서의 크기를 확인합니다.
#
# 예상 출력:
# TensorShape([2, 70, 120, 16])
pooled_fmaps.shape

In [ ]:
# 전역 평균 풀링(Global Average Pooling) 층을 생성합니다.
#
# GlobalAvgPool2D는 GlobalAveragePooling2D의 별칭(alias)입니다.
# 즉, 아래 두 코드는 같은 의미입니다.
#
# tf.keras.layers.GlobalAvgPool2D()
# tf.keras.layers.GlobalAveragePooling2D()
#
# 전역 평균 풀링은 각 특성 맵(feature map)의 공간 방향,
# 즉 높이(height)와 너비(width) 방향 전체에 대해 평균을 계산합니다.
global_avg_pool = tf.keras.layers.GlobalAvgPool2D()


# 전역 평균 풀링의 입력 텐서는 일반적으로 다음과 같은 4차원 형태입니다.
#
# (배치 크기, 높이, 너비, 채널 수)
#
# 여기서 images의 shape은 앞에서 확인한 것처럼 다음과 같습니다.
#
# (2, 70, 120, 3)
#
# 의미:
# - 2: 이미지 개수
# - 70: 이미지 높이
# - 120: 이미지 너비
# - 3: RGB 색상 채널
#
# GlobalAvgPool2D는 각 이미지마다,
# 각 채널별로 높이와 너비 전체의 평균값을 계산합니다.
#
# 따라서 입력 shape:
# (2, 70, 120, 3)
#
# 출력 shape:
# (2, 3)
#
# 즉, 이미지 1장마다 RGB 채널별 평균값 3개가 출력됩니다.
global_avg_pool(images)


# 위의 GlobalAvgPool2D 층은 Lambda 층으로도 직접 구현할 수 있습니다.
#
# Lambda 층은 사용자가 원하는 TensorFlow 연산을
# Keras 층처럼 사용할 수 있게 해 주는 층입니다.
global_avg_pool = tf.keras.layers.Lambda(
    lambda X: tf.reduce_mean(
        X,              # 입력 텐서입니다.
                        # 예: shape이 (2, 70, 120, 3)인 이미지 텐서

        axis=[1, 2]     # 평균을 계산할 축을 지정합니다.
                        #
                        # axis=1은 높이(height) 방향입니다.
                        # axis=2는 너비(width) 방향입니다.
                        #
                        # 즉, 각 이미지의 각 채널마다
                        # 모든 픽셀 위치의 평균을 계산합니다.
                        #
                        # 배치 축(axis=0)은 유지됩니다.
                        # 채널 축(axis=3)도 유지됩니다.
    )
)


# Lambda 층으로 만든 전역 평균 풀링을 images에 적용합니다.
#
# 입력 images의 shape:
# (2, 70, 120, 3)
#
# 출력 결과의 shape:
# (2, 3)
#
# 의미:
# - 2: 이미지 2장
# - 3: 각 이미지의 RGB 채널 평균값
#
# 출력값 예:
# [[0.64338624, 0.5971759 , 0.5824972 ],
#  [0.76306933, 0.26011038, 0.10849128]]
#
# 첫 번째 행은 첫 번째 이미지의
# 빨강(R), 초록(G), 파랑(B) 채널 평균 강도입니다.
#
# 두 번째 행은 두 번째 이미지의
# 빨강(R), 초록(G), 파랑(B) 채널 평균 강도입니다.
#
# 앞에서 Rescaling(scale=1 / 255)을 적용했기 때문에
# 픽셀값은 0~255가 아니라 0~1 범위입니다.
global_avg_pool(images)


# 결과의 크기만 확인하고 싶다면 shape을 사용할 수 있습니다.
#
# 예상 출력:
# TensorShape([2, 3])
global_avg_pool(images).shape

In [ ]:
# functools 모듈에서 partial 함수를 import합니다.
# partial은 어떤 함수의 일부 인수를 미리 고정해 둔 새 함수를 만들 때 사용합니다.
#
# 여기서는 Conv2D 층을 만들 때 자주 반복되는 옵션들을
# 매번 쓰지 않기 위해 partial을 사용합니다.
from functools import partial

# TensorFlow 라이브러리 import
import tensorflow as tf


# DefaultConv2D는 기본 설정이 미리 지정된 Conv2D 층 생성 함수입니다.
#
# 원래 Conv2D를 만들 때마다 다음 옵션들을 반복해서 써야 합니다.
#
# kernel_size=3
# padding="same"
# activation="relu"
# kernel_initializer="he_normal"
#
# partial을 사용하면 이 옵션들을 미리 고정해 둘 수 있습니다.
# 이후 DefaultConv2D(filters=128)처럼 필요한 값만 추가로 넘기면 됩니다.
DefaultConv2D = partial(
    tf.keras.layers.Conv2D,

    kernel_size=3,              # 기본 커널 크기를 3x3으로 설정합니다.
                                # 3x3 커널은 CNN에서 가장 자주 사용되는 크기입니다.

    padding="same",             # 입력과 출력의 높이, 너비가 같도록 패딩을 추가합니다.
                                # 예: 28x28 입력 → 28x28 출력

    activation="relu",          # 활성화 함수로 ReLU를 사용합니다.
                                # ReLU는 음수는 0으로 만들고 양수는 그대로 통과시킵니다.

    kernel_initializer="he_normal"
                                # He 정규 초기화를 사용합니다.
                                # ReLU 활성화 함수와 함께 자주 사용되는 초기화 방법입니다.
)


# 패션 MNIST 데이터셋을 분류하기 위한 기본 CNN 모델을 만듭니다.
#
# Fashion MNIST 이미지는 28x28 크기의 흑백 이미지입니다.
# 따라서 입력 shape은 [28, 28, 1]입니다.
#
# 의미:
# - 28: 이미지 높이
# - 28: 이미지 너비
# - 1: 흑백 이미지이므로 채널 수 1개
#
# 모델 구조는 크게 다음 순서로 구성됩니다.
#
# 1. 합성곱 층과 최대 풀링 층으로 이미지 특징 추출
# 2. Flatten으로 1차원 벡터로 변환
# 3. Dense 층으로 분류 수행
# 4. Dropout으로 과대적합 감소
# 5. softmax 출력층으로 10개 클래스 확률 출력
model = tf.keras.Sequential([

    # 첫 번째 합성곱 층입니다.
    # 입력 이미지에서 64개의 특징 맵을 추출합니다.
    #
    # filters=64:
    # - 64개의 필터를 사용합니다.
    # - 출력 채널 수가 64가 됩니다.
    #
    # kernel_size=7:
    # - 이 층에서는 DefaultConv2D의 기본값인 3x3 대신 7x7 커널을 사용합니다.
    # - 첫 층에서 비교적 넓은 영역을 보며 저수준 특징을 추출합니다.
    #
    # input_shape=[28, 28, 1]:
    # - Fashion MNIST 입력 이미지의 형태를 지정합니다.
    # - 28x28 흑백 이미지입니다.
    #
    # padding="same"이므로 출력 높이와 너비는 28x28로 유지됩니다.
    DefaultConv2D(
        filters=64,
        kernel_size=7,
        input_shape=[28, 28, 1]
    ),

    # 최대 풀링 층입니다.
    # 기본 pool_size는 2이므로 2x2 영역마다 최댓값을 하나 선택합니다.
    #
    # 높이와 너비가 절반으로 줄어듭니다.
    #
    # 입력:
    # 28x28x64
    #
    # 출력:
    # 14x14x64
    tf.keras.layers.MaxPool2D(),

    # 두 번째 합성곱 층입니다.
    # 128개의 필터를 사용하여 더 많은 특징을 추출합니다.
    #
    # DefaultConv2D의 기본 설정이 적용됩니다.
    # - kernel_size=3
    # - padding="same"
    # - activation="relu"
    # - kernel_initializer="he_normal"
    #
    # 입력:
    # 14x14x64
    #
    # 출력:
    # 14x14x128
    DefaultConv2D(filters=128),

    # 세 번째 합성곱 층입니다.
    # 같은 해상도에서 128개의 특징 맵을 한 번 더 학습합니다.
    #
    # 여러 합성곱 층을 연속으로 쌓으면
    # 더 복잡하고 추상적인 패턴을 학습할 수 있습니다.
    #
    # 출력:
    # 14x14x128
    DefaultConv2D(filters=128),

    # 두 번째 최대 풀링 층입니다.
    #
    # 입력:
    # 14x14x128
    #
    # 출력:
    # 7x7x128
    tf.keras.layers.MaxPool2D(),

    # 네 번째 합성곱 층입니다.
    # 필터 수를 256개로 늘립니다.
    #
    # 일반적으로 공간 크기가 줄어들수록
    # 채널 수, 즉 필터 수를 늘려 더 풍부한 특징을 학습하게 합니다.
    #
    # 입력:
    # 7x7x128
    #
    # 출력:
    # 7x7x256
    DefaultConv2D(filters=256),

    # 다섯 번째 합성곱 층입니다.
    # 256개의 필터를 한 번 더 적용합니다.
    #
    # 출력:
    # 7x7x256
    DefaultConv2D(filters=256),

    # 세 번째 최대 풀링 층입니다.
    #
    # 입력:
    # 7x7x256
    #
    # 출력:
    # 3x3x256
    #
    # 7x7에 기본 MaxPool2D(pool_size=2)를 적용하면
    # padding 기본값이 "valid"이므로 3x3이 됩니다.
    tf.keras.layers.MaxPool2D(),

    # Flatten 층은 다차원 특성 맵을 1차원 벡터로 펼칩니다.
    #
    # 입력:
    # 3x3x256
    #
    # 출력:
    # 3 * 3 * 256 = 2304개의 값을 가진 벡터
    #
    # 이 벡터가 이후 Dense 층의 입력으로 사용됩니다.
    tf.keras.layers.Flatten(),

    # 완전 연결층(Dense layer)입니다.
    # CNN이 추출한 특징을 바탕으로 분류에 필요한 조합을 학습합니다.
    #
    # units=128:
    # - 뉴런 128개를 사용합니다.
    #
    # activation="relu":
    # - ReLU 활성화 함수를 사용합니다.
    #
    # kernel_initializer="he_normal":
    # - ReLU와 잘 맞는 He 정규 초기화를 사용합니다.
    tf.keras.layers.Dense(
        units=128,
        activation="relu",
        kernel_initializer="he_normal"
    ),

    # Dropout 층입니다.
    #
    # 훈련 중에 뉴런의 50%를 무작위로 꺼서
    # 특정 뉴런에 지나치게 의존하는 것을 방지합니다.
    #
    # 이를 통해 과대적합(overfitting)을 줄이는 데 도움을 줍니다.
    #
    # 주의:
    # Dropout은 훈련할 때만 적용되고,
    # 평가나 예측할 때는 적용되지 않습니다.
    tf.keras.layers.Dropout(0.5),

    # 두 번째 완전 연결층입니다.
    #
    # units=64:
    # - 뉴런 64개를 사용합니다.
    #
    # 앞의 Dense 층보다 작은 크기로,
    # 분류에 필요한 핵심 정보를 더 압축해서 학습합니다.
    tf.keras.layers.Dense(
        units=64,
        activation="relu",
        kernel_initializer="he_normal"
    ),

    # 두 번째 Dropout 층입니다.
    # 다시 50%의 뉴런을 무작위로 비활성화하여
    # 과대적합을 줄입니다.
    tf.keras.layers.Dropout(0.5),

    # 출력층입니다.
    #
    # Fashion MNIST는 총 10개의 클래스를 분류하는 문제입니다.
    # 예:
    # 티셔츠, 바지, 풀오버, 드레스, 코트,
    # 샌들, 셔츠, 운동화, 가방, 앵클부츠
    #
    # units=10:
    # - 클래스가 10개이므로 출력 뉴런도 10개입니다.
    #
    # activation="softmax":
    # - 각 클래스에 대한 확률을 출력합니다.
    # - 출력값 10개의 합은 1이 됩니다.
    tf.keras.layers.Dense(
        units=10,
        activation="softmax"
    )

])


# 모델 구조를 요약해서 확인합니다.
# 각 층의 출력 shape과 파라미터 수를 볼 수 있습니다.
model.summary()

In [ ]:
# functools 모듈에서 partial 함수를 import합니다.
# partial은 어떤 함수의 일부 인수를 미리 고정해 둔 새 함수를 만들 때 사용합니다.
#
# 여기서는 Conv2D 층을 만들 때 반복해서 사용하는 옵션들을
# DefaultConv2D라는 이름으로 미리 묶어 두기 위해 사용합니다.
from functools import partial

# TensorFlow 라이브러리 import
import tensorflow as tf


# Conv2D 층의 기본 설정을 미리 지정합니다.
#
# ResNet에서는 3x3 합성곱을 많이 사용하므로
# kernel_size=3을 기본값으로 둡니다.
#
# padding="same"은 입력과 출력의 높이, 너비를 같게 유지하기 위해 사용합니다.
#
# kernel_initializer="he_normal"은 ReLU 활성화 함수와 잘 어울리는
# He 정규 초기화 방법입니다.
#
# use_bias=False는 BatchNormalization을 사용할 때 자주 쓰는 설정입니다.
# BatchNormalization 층이 자체적으로 이동 파라미터를 가지므로
# Conv2D의 bias는 생략해도 됩니다.
DefaultConv2D = partial(
    tf.keras.layers.Conv2D,
    kernel_size=3,
    strides=1,
    padding="same",
    kernel_initializer="he_normal",
    use_bias=False
)


# ResidualUnit은 ResNet의 핵심 블록입니다.
#
# 일반적인 신경망 층은 입력 X를 여러 층에 통과시켜 F(X)를 만듭니다.
#
# 하지만 ResNet의 Residual Unit은 다음과 같이 계산합니다.
#
# 출력 = F(X) + X
#
# 즉, 입력 X를 변환한 결과 F(X)에
# 원래 입력 X를 그대로 더합니다.
#
# 이때 입력을 그대로 더하는 경로를 skip connection,
# shortcut connection, 또는 identity connection이라고 부릅니다.
#
# 이런 구조는 깊은 신경망에서 그래디언트가 더 잘 흐르도록 도와줍니다.
class ResidualUnit(tf.keras.layers.Layer):

    # filters:
    #   이 Residual Unit에서 사용할 필터 개수입니다.
    #   출력 feature map의 채널 수가 됩니다.
    #
    # strides:
    #   첫 번째 합성곱 층에서 사용할 stride입니다.
    #   strides=1이면 높이와 너비가 유지됩니다.
    #   strides=2이면 높이와 너비가 절반으로 줄어듭니다.
    #
    # activation:
    #   사용할 활성화 함수입니다.
    #   ResNet에서는 보통 ReLU를 사용합니다.
    #
    # **kwargs:
    #   name, dtype 같은 Keras 층 공통 인수를 받을 수 있게 합니다.
    def __init__(self, filters, strides=1, activation="relu", **kwargs):

        # 부모 클래스인 tf.keras.layers.Layer의 초기화 메서드를 호출합니다.
        super().__init__(**kwargs)

        # activation 문자열을 실제 Keras 활성화 함수로 변환합니다.
        #
        # 예:
        # activation="relu"
        # → tf.keras.activations.relu 함수
        self.activation = tf.keras.activations.get(activation)

        # main_layers는 입력이 실제로 통과하는 주 경로입니다.
        #
        # 이 구조는 다음 순서로 이루어져 있습니다.
        #
        # 1. Conv2D
        # 2. BatchNormalization
        # 3. ReLU
        # 4. Conv2D
        # 5. BatchNormalization
        #
        # 마지막에는 바로 ReLU를 적용하지 않습니다.
        # 먼저 skip connection의 결과와 더한 뒤 ReLU를 적용합니다.
        self.main_layers = [

            # 첫 번째 3x3 합성곱 층입니다.
            #
            # strides가 1이면 공간 크기를 유지합니다.
            # strides가 2이면 높이와 너비를 절반으로 줄입니다.
            #
            # 예:
            # 입력: 56x56x64
            # strides=2
            # 출력: 28x28x128
            DefaultConv2D(
                filters,
                strides=strides
            ),

            # BatchNormalization 층입니다.
            #
            # 각 미니배치마다 출력을 정규화하여
            # 훈련을 더 안정적으로 만들어 줍니다.
            tf.keras.layers.BatchNormalization(),

            # ReLU 활성화 함수입니다.
            #
            # 음수 값은 0으로 만들고,
            # 양수 값은 그대로 통과시킵니다.
            tf.keras.layers.Activation(self.activation),

            # 두 번째 3x3 합성곱 층입니다.
            #
            # 여기서는 strides를 따로 지정하지 않았으므로
            # DefaultConv2D의 기본값인 strides=1이 사용됩니다.
            #
            # 따라서 두 번째 합성곱은 공간 크기를 유지합니다.
            DefaultConv2D(filters),

            # 두 번째 BatchNormalization 층입니다.
            #
            # 이 뒤에서 skip connection 결과와 더한 다음
            # 최종 ReLU를 적용합니다.
            tf.keras.layers.BatchNormalization()
        ]

        # skip_layers는 스킵 연결 경로입니다.
        #
        # 기본적으로 skip connection은 입력을 그대로 사용합니다.
        #
        # 즉:
        # skip_Z = inputs
        #
        # 하지만 strides > 1인 경우에는 문제가 생깁니다.
        #
        # main path는 strides=2 때문에 높이와 너비가 줄어들고,
        # filters 값에 따라 채널 수도 바뀔 수 있습니다.
        #
        # 이 상태에서는 Z + skip_Z를 할 수 없습니다.
        # 두 텐서의 shape이 다르기 때문입니다.
        #
        # 그래서 strides > 1일 때는 skip path에도 1x1 합성곱을 적용하여
        # 높이, 너비, 채널 수를 main path 출력과 맞춰 줍니다.
        self.skip_layers = []

        if strides > 1:
            self.skip_layers = [

                # 1x1 합성곱은 공간적인 3x3 패턴을 보는 목적이 아니라,
                # 채널 수와 공간 크기를 맞추기 위해 사용됩니다.
                #
                # kernel_size=1:
                #   1x1 합성곱을 사용합니다.
                #
                # strides=strides:
                #   main path와 같은 비율로 높이와 너비를 줄입니다.
                #
                # filters:
                #   main path와 같은 채널 수로 맞춥니다.
                DefaultConv2D(
                    filters,
                    kernel_size=1,
                    strides=strides
                ),

                # skip path에도 BatchNormalization을 적용합니다.
                tf.keras.layers.BatchNormalization()
            ]

    # call()은 이 층이 실제 입력 데이터를 받을 때 실행되는 메서드입니다.
    #
    # 예:
    # Z = ResidualUnit(64)(inputs)
    #
    # 위 코드가 실행되면 내부적으로 call(inputs)가 호출됩니다.
    def call(self, inputs):

        # 먼저 main path의 입력으로 원본 inputs를 사용합니다.
        Z = inputs

        # main_layers에 들어 있는 층들을 순서대로 통과시킵니다.
        #
        # 흐름:
        # Conv2D → BatchNormalization → ReLU
        # → Conv2D → BatchNormalization
        for layer in self.main_layers:
            Z = layer(Z)

        # skip path는 기본적으로 입력을 그대로 사용합니다.
        skip_Z = inputs

        # 만약 strides > 1이었다면,
        # self.skip_layers 안에 1x1 Conv2D와 BatchNormalization이 들어 있습니다.
        #
        # 이 경우 skip_Z도 main path의 출력 Z와 같은 shape이 되도록 변환됩니다.
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)

        # main path의 출력 Z와 skip path의 출력 skip_Z를 더합니다.
        #
        # 이것이 ResNet의 핵심인 residual connection입니다.
        #
        # 수식으로 표현하면:
        #
        # output = activation(F(X) + X)
        #
        # 단, shape이 다를 때는 X를 그대로 더하지 않고
        # 1x1 합성곱으로 변환한 값을 더합니다.
        return self.activation(Z + skip_Z)

In [ ]:
# TensorFlow 라이브러리 import
import tensorflow as tf

# partial은 함수의 일부 인수를 미리 고정한 새 함수를 만들 때 사용합니다.
# 여기서는 Conv2D 층을 만들 때 반복되는 기본 옵션을 줄이기 위해 사용합니다.
from functools import partial


# Conv2D 층의 기본 설정을 미리 정의합니다.
#
# ResNet에서는 3x3 합성곱을 많이 사용하므로 kernel_size=3을 기본값으로 둡니다.
#
# padding="same":
# - 입력과 출력의 높이, 너비를 같게 유지하도록 패딩을 추가합니다.
#
# kernel_initializer="he_normal":
# - ReLU 활성화 함수와 잘 어울리는 He 정규 초기화 방법입니다.
#
# use_bias=False:
# - Conv2D 뒤에 BatchNormalization을 사용할 것이므로 bias를 생략합니다.
# - BatchNormalization이 자체적으로 이동 파라미터를 가지기 때문에
#   Conv2D의 bias는 보통 필요하지 않습니다.
DefaultConv2D = partial(
    tf.keras.layers.Conv2D,
    kernel_size=3,
    strides=1,
    padding="same",
    kernel_initializer="he_normal",
    use_bias=False
)


# ResNet의 기본 블록인 Residual Unit을 정의합니다.
#
# 이 층은 다음 구조를 가집니다.
#
# main path:
# Conv2D → BatchNormalization → ReLU → Conv2D → BatchNormalization
#
# skip path:
# 기본적으로 입력을 그대로 통과
# 단, strides > 1이면 1x1 Conv2D로 크기와 채널 수를 맞춤
#
# 최종 출력:
# ReLU(main path 출력 + skip path 출력)
class ResidualUnit(tf.keras.layers.Layer):

    def __init__(self, filters, strides=1, activation="relu", **kwargs):

        # Keras Layer 부모 클래스 초기화
        super().__init__(**kwargs)

        # 문자열로 받은 활성화 함수를 실제 함수로 변환합니다.
        # 예: "relu" → tf.keras.activations.relu
        self.activation = tf.keras.activations.get(activation)

        # main path에 들어갈 층들을 리스트로 저장합니다.
        self.main_layers = [

            # 첫 번째 3x3 합성곱 층입니다.
            #
            # strides=1이면 높이와 너비가 유지됩니다.
            # strides=2이면 높이와 너비가 절반으로 줄어듭니다.
            #
            # 필터 수는 이 Residual Unit의 출력 채널 수가 됩니다.
            DefaultConv2D(
                filters,
                strides=strides
            ),

            # 첫 번째 배치 정규화 층입니다.
            # 합성곱 결과를 정규화하여 학습을 안정적으로 만듭니다.
            tf.keras.layers.BatchNormalization(),

            # ReLU 활성화 함수입니다.
            tf.keras.layers.Activation(self.activation),

            # 두 번째 3x3 합성곱 층입니다.
            #
            # 여기서는 strides를 지정하지 않으므로 기본값인 strides=1이 사용됩니다.
            # 따라서 공간 크기를 추가로 줄이지 않습니다.
            DefaultConv2D(filters),

            # 두 번째 배치 정규화 층입니다.
            #
            # 이 뒤에서 skip path와 더한 후 최종 ReLU를 적용합니다.
            tf.keras.layers.BatchNormalization()
        ]

        # skip path에 들어갈 층들을 저장할 리스트입니다.
        #
        # 기본적으로는 빈 리스트입니다.
        # 즉, 입력을 그대로 skip_Z로 사용합니다.
        self.skip_layers = []

        # strides > 1이면 main path에서 높이와 너비가 줄어듭니다.
        #
        # 예:
        # 입력: 56x56x64
        # main path 출력: 28x28x128
        #
        # 이 경우 입력을 그대로 더하면 shape이 맞지 않아 오류가 납니다.
        #
        # 따라서 skip path에도 1x1 합성곱을 적용해
        # main path 출력과 같은 shape으로 맞춰 줍니다.
        if strides > 1:
            self.skip_layers = [

                # 1x1 합성곱입니다.
                #
                # 목적:
                # - 높이와 너비를 strides에 맞게 줄임
                # - 채널 수를 filters에 맞춤
                #
                # 3x3 합성곱처럼 주변 공간 패턴을 많이 보는 것이 아니라,
                # 주로 shape을 맞추는 역할을 합니다.
                DefaultConv2D(
                    filters,
                    kernel_size=1,
                    strides=strides
                ),

                # skip path에도 BatchNormalization을 적용합니다.
                tf.keras.layers.BatchNormalization()
            ]

    def call(self, inputs):

        # main path 계산을 시작합니다.
        Z = inputs

        # main_layers 안의 층들을 순서대로 통과시킵니다.
        #
        # Conv2D → BatchNormalization → ReLU
        # → Conv2D → BatchNormalization
        for layer in self.main_layers:
            Z = layer(Z)

        # skip path 계산입니다.
        #
        # 기본값은 입력을 그대로 사용하는 것입니다.
        skip_Z = inputs

        # strides > 1인 경우에는 skip_layers에
        # 1x1 Conv2D와 BatchNormalization이 들어 있으므로
        # 이를 통과해 shape을 맞춥니다.
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)

        # main path의 출력과 skip path의 출력을 더합니다.
        #
        # 이것이 ResNet의 핵심인 skip connection입니다.
        #
        # 그 뒤 ReLU 활성화 함수를 적용합니다.
        return self.activation(Z + skip_Z)


# Sequential 클래스를 사용해 ResNet-34와 비슷한 구조의 모델을 만듭니다.
#
# 입력 이미지는 ImageNet 스타일의 224x224 RGB 이미지라고 가정합니다.
#
# 입력 shape:
# [224, 224, 3]
#
# 의미:
# - 224: 이미지 높이
# - 224: 이미지 너비
# - 3: RGB 색상 채널
model = tf.keras.Sequential([

    # 첫 번째 합성곱 층입니다.
    #
    # ResNet의 시작 부분에서는 큰 7x7 커널을 사용합니다.
    #
    # filters=64:
    # - 64개의 특징 맵을 만듭니다.
    #
    # kernel_size=7:
    # - 7x7 크기의 큰 필터를 사용합니다.
    #
    # strides=2:
    # - 높이와 너비를 절반으로 줄입니다.
    #
    # input_shape=[224, 224, 3]:
    # - 입력 이미지 크기를 지정합니다.
    #
    # 출력 크기:
    # 224x224x3 → 112x112x64
    DefaultConv2D(
        64,
        kernel_size=7,
        strides=2,
        input_shape=[224, 224, 3]
    ),

    # 배치 정규화 층입니다.
    #
    # 첫 번째 합성곱의 출력을 정규화하여 학습을 안정화합니다.
    tf.keras.layers.BatchNormalization(),

    # ReLU 활성화 함수입니다.
    #
    # 음수 값은 0으로 만들고,
    # 양수 값은 그대로 통과시킵니다.
    tf.keras.layers.Activation("relu"),

    # 최대 풀링 층입니다.
    #
    # pool_size=3:
    # - 3x3 영역에서 최댓값을 선택합니다.
    #
    # strides=2:
    # - 높이와 너비를 다시 절반 정도로 줄입니다.
    #
    # padding="same":
    # - 출력 크기를 적절히 유지하기 위해 가장자리에 패딩을 추가합니다.
    #
    # 출력 크기:
    # 112x112x64 → 56x56x64
    tf.keras.layers.MaxPool2D(
        pool_size=3,
        strides=2,
        padding="same"
    )
])


# prev_filters는 이전 Residual Unit의 필터 수를 기억하기 위한 변수입니다.
#
# 처음 합성곱 층의 출력 채널 수가 64이므로
# 시작값을 64로 설정합니다.
prev_filters = 64


# ResNet-34는 residual block을 다음 개수만큼 사용합니다.
#
# 64 필터 블록  : 3개
# 128 필터 블록 : 4개
# 256 필터 블록 : 6개
# 512 필터 블록 : 3개
#
# 이를 리스트로 표현하면 다음과 같습니다.
#
# [64] * 3   → [64, 64, 64]
# [128] * 4  → [128, 128, 128, 128]
# [256] * 6  → [256, 256, 256, 256, 256, 256]
# [512] * 3  → [512, 512, 512]
#
# 모두 합치면 총 16개의 Residual Unit이 됩니다.
#
# 각 Residual Unit은 Conv2D를 2개씩 가지므로
# residual 내부 합성곱 층은 16 x 2 = 32개입니다.
#
# 처음의 7x7 Conv2D 1개와 마지막 Dense 층 1개를 합쳐
# ResNet-34라고 부릅니다.
for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:

    # 필터 수가 이전 블록과 같으면 spatial size를 유지합니다.
    #
    # 예:
    # 이전 filters=64, 현재 filters=64
    # → strides=1
    #
    # 입력과 출력의 높이, 너비가 같습니다.
    #
    # 필터 수가 바뀌는 첫 블록에서는 strides=2를 사용합니다.
    #
    # 예:
    # 이전 filters=64, 현재 filters=128
    # → strides=2
    #
    # 이때 높이와 너비를 절반으로 줄이고,
    # 채널 수를 128로 늘립니다.
    strides = 1 if filters == prev_filters else 2

    # Residual Unit을 모델에 추가합니다.
    #
    # filters:
    # - 출력 채널 수
    #
    # strides:
    # - 같은 stage 안에서는 1
    # - 새로운 stage로 넘어갈 때는 2
    model.add(
        ResidualUnit(
            filters,
            strides=strides
        )
    )

    # 현재 필터 수를 다음 반복에서 이전 필터 수로 사용하기 위해 저장합니다.
    prev_filters = filters


# 전역 평균 풀링 층입니다.
#
# 각 feature map의 높이와 너비 방향 평균을 계산합니다.
#
# 예:
# 입력 shape:
# (batch_size, 7, 7, 512)
#
# 출력 shape:
# (batch_size, 512)
#
# 즉, 각 이미지마다 512개의 평균값을 가진 벡터가 됩니다.
model.add(tf.keras.layers.GlobalAvgPool2D())


# Flatten 층입니다.
#
# GlobalAvgPool2D의 출력은 이미 보통 2차원입니다.
#
# 예:
# (batch_size, 512)
#
# 따라서 이 코드에서는 Flatten이 꼭 필요하지는 않지만,
# 책의 코드 흐름에 맞춰 추가할 수 있습니다.
#
# Flatten을 통과해도 shape은 그대로 유지됩니다.
#
# (batch_size, 512) → (batch_size, 512)
model.add(tf.keras.layers.Flatten())


# 출력층입니다.
#
# units=10:
# - 10개 클래스를 분류한다고 가정합니다.
# - 예를 들어 CIFAR-10처럼 클래스가 10개인 데이터셋에 사용할 수 있습니다.
#
# activation="softmax":
# - 각 클래스에 대한 확률을 출력합니다.
# - 출력값 10개의 합은 1입니다.
#
# 만약 ImageNet처럼 1000개 클래스를 분류한다면
# units=1000으로 바꾸면 됩니다.
model.add(
    tf.keras.layers.Dense(
        10,
        activation="softmax"
    )
)


# 모델 구조를 요약해서 확인합니다.
#
# 각 층의 출력 shape과 파라미터 수를 볼 수 있습니다.
model.summary()

In [ ]:
# 사이킷런에서 제공하는 샘플 이미지 2장을 불러오기 위한 함수입니다.
from sklearn.datasets import load_sample_images

# TensorFlow 라이브러리 import
import tensorflow as tf


# tf.keras.applications 패키지에는 여러 사전 훈련된 모델이 준비되어 있습니다.
#
# 예:
# - ResNet50
# - VGG16
# - VGG19
# - Xception
# - InceptionV3
# - MobileNet
# - EfficientNet 등
#
# 여기서는 ImageNet 데이터셋으로 미리 훈련된 ResNet-50 모델을 불러옵니다.
#
# weights="imagenet":
# - ImageNet 데이터셋으로 학습된 가중치를 사용하겠다는 뜻입니다.
#
# ImageNet은 1000개의 클래스를 가진 대규모 이미지 분류 데이터셋입니다.
# 따라서 이 모델의 최종 출력은 1000개 클래스에 대한 확률입니다.
model = tf.keras.applications.resnet50.ResNet50(
    weights="imagenet"
)


# load_sample_images()는 사이킷런에 내장된 샘플 이미지 2장을 불러옵니다.
#
# 반환값은 딕셔너리 형태이며,
# 실제 이미지 배열은 "images" 키에 들어 있습니다.
#
# 원본 images의 shape은 보통 다음과 같습니다.
#
# (2, 427, 640, 3)
#
# 의미:
# - 2: 이미지 개수
# - 427: 이미지 높이
# - 640: 이미지 너비
# - 3: RGB 색상 채널
#
# 픽셀값은 일반적으로 0~255 범위의 정수입니다.
images = load_sample_images()["images"]


# ResNet-50은 기본적으로 224x224 크기의 RGB 이미지를 입력으로 받도록 설계되어 있습니다.
#
# 따라서 샘플 이미지의 크기를 224x224로 변경합니다.
#
# Resizing 층은 이미지 크기를 바꾸는 Keras 전처리 층입니다.
images_resized = tf.keras.layers.Resizing(
    height=224,                 # 출력 이미지의 높이를 224픽셀로 지정합니다.
    width=224,                  # 출력 이미지의 너비를 224픽셀로 지정합니다.

    crop_to_aspect_ratio=True   # 원본 이미지의 가로세로 비율을 최대한 유지합니다.
                                #
                                # True로 설정하면 단순히 찌그러뜨려 224x224로 만드는 것이 아니라,
                                # 비율을 맞추기 위해 필요한 부분을 잘라낸 뒤 크기를 조정합니다.
                                #
                                # 즉, 이미지가 심하게 왜곡되는 것을 줄일 수 있습니다.
)(images)


# ResNet-50에 입력하기 전에 전처리를 수행합니다.
#
# preprocess_input()은 ResNet-50이 학습될 때 사용된 입력 형식에 맞게
# 이미지 픽셀값을 변환해 주는 함수입니다.
#
# 중요한 점:
# - 여기서는 입력 이미지의 픽셀값이 0~255 범위라고 가정합니다.
# - 앞에서 Rescaling(1 / 255)을 적용하면 안 됩니다.
#
# tf.keras.applications.resnet50.preprocess_input()은 ResNet-50용 전처리를 수행합니다.
# 구체적으로는 RGB 이미지를 모델이 기대하는 형식으로 변환하고,
# ImageNet 학습 방식에 맞는 평균값 보정 등을 적용합니다.
inputs = tf.keras.applications.resnet50.preprocess_input(
    images_resized
)


# 사전 훈련된 ResNet-50 모델을 사용해 예측을 수행합니다.
#
# inputs의 shape:
# (2, 224, 224, 3)
#
# 의미:
# - 2: 이미지 2장
# - 224: 이미지 높이
# - 224: 이미지 너비
# - 3: RGB 색상 채널
#
# model.predict(inputs)는 각 이미지가 ImageNet의 1000개 클래스 중
# 어느 클래스에 속할 확률이 높은지 계산합니다.
Y_proba = model.predict(inputs)


# 예측 결과의 shape을 확인합니다.
#
# 출력 결과:
# (2, 1000)
#
# 의미:
# - 2: 이미지 2장에 대한 예측 결과
# - 1000: ImageNet의 1000개 클래스 각각에 대한 확률
#
# 즉, 첫 번째 행은 첫 번째 이미지의 1000개 클래스 확률이고,
# 두 번째 행은 두 번째 이미지의 1000개 클래스 확률입니다.
Y_proba.shape

In [ ]:
# decode_predictions() 함수는 ResNet-50 모델의 예측 결과를
# 사람이 읽기 쉬운 클래스 이름으로 변환해 줍니다.
#
# model.predict(inputs)의 결과인 Y_proba는 다음과 같은 shape을 가집니다.
#
# (2, 1000)
#
# 의미:
# - 2: 이미지 2장
# - 1000: ImageNet의 1000개 클래스에 대한 예측 확률
#
# 하지만 Y_proba 자체에는 클래스 이름이 들어 있지 않고,
# 각 클래스 번호에 대한 확률값만 들어 있습니다.
#
# decode_predictions()를 사용하면
# 확률이 가장 높은 클래스들을 클래스 ID, 클래스 이름, 확률로 변환할 수 있습니다.
top_K = tf.keras.applications.resnet50.decode_predictions(
    Y_proba,   # ResNet-50 모델이 출력한 예측 확률입니다.
               # shape은 (이미지 개수, 1000)입니다.

    top=3      # 각 이미지마다 확률이 가장 높은 상위 3개 클래스를 가져옵니다.
)


# len(images)는 입력 이미지의 개수를 의미합니다.
#
# 여기서는 load_sample_images()로 샘플 이미지 2장을 불러왔으므로
# len(images)는 2입니다.
#
# 따라서 image_index는 0, 1 순서로 반복됩니다.
for image_index in range(len(images)):

    # 현재 몇 번째 이미지의 예측 결과인지 출력합니다.
    #
    # f-string을 사용하여 image_index 값을 문자열 안에 넣습니다.
    #
    # 출력 예:
    # Image #0
    # Image #1
    print(f"Image #{image_index}")

    # top_K[image_index]에는 해당 이미지에 대한
    # 상위 3개 예측 결과가 들어 있습니다.
    #
    # 각 예측 결과는 다음 3개의 값으로 이루어진 튜플입니다.
    #
    # class_id:
    #   ImageNet 클래스 ID입니다.
    #   예: "n03877845"
    #
    # name:
    #   클래스 이름입니다.
    #   예: "palace", "monastery"
    #
    # y_proba:
    #   해당 클래스일 것으로 예측한 확률입니다.
    #   예: 0.5469
    #
    # top=3으로 설정했으므로
    # 각 이미지마다 3번 반복됩니다.
    for class_id, name, y_proba in top_K[image_index]:

        # 예측 결과를 보기 좋게 출력합니다.
        #
        # f" {class_id} - {name:12s} {y_proba:.2%}"
        #
        # class_id:
        #   ImageNet 클래스 ID를 그대로 출력합니다.
        #
        # name:12s:
        #   클래스 이름을 문자열로 출력하되,
        #   최소 12칸의 공간을 확보합니다.
        #
        #   예:
        #   "palace"는 글자 수가 6개이므로
        #   뒤에 공백이 추가되어 정렬됩니다.
        #
        # y_proba:.2%:
        #   예측 확률을 퍼센트 형식으로 출력합니다.
        #   소수점 아래 둘째 자리까지 표시합니다.
        #
        #   예:
        #   0.5469 → 54.69%
        #
        # 출력 예:
        # n03877845 - palace       54.69%
        print(f" {class_id} - {name:12s} {y_proba:.2%}")


# 예상 출력 예시는 다음과 비슷합니다.
#
# Image #0
#  n03877845 - palace       54.69%
#  n03781244 - monastery    24.72%
#  n02825657 - bell_cote    18.55%
#
# Image #1
#  n04522168 - vase         32.66%
#  n11939491 - daisy        17.81%
#  n03530642 - honeycomb    12.06%
#
# 주의:
# TensorFlow/Keras 버전이나 실행 환경에 따라
# 확률값이 아주 조금 다르게 나올 수 있습니다.

In [ ]:
# TensorFlow Datasets 라이브러리를 import합니다.
#
# tensorflow_datasets는 TensorFlow에서 바로 사용할 수 있는
# 여러 공개 데이터셋을 쉽게 불러올 수 있게 해 주는 라이브러리입니다.
#
# 여기서는 꽃 이미지 데이터셋인 tf_flowers를 사용합니다.
import tensorflow_datasets as tfds


# tfds.load()를 사용해 tf_flowers 데이터셋을 불러옵니다.
#
# "tf_flowers":
# - TensorFlow Datasets에 포함된 꽃 이미지 분류 데이터셋입니다.
# - 총 5개의 꽃 클래스로 이루어져 있습니다.
#
# as_supervised=True:
# - 데이터셋을 (이미지, 레이블) 형태로 반환합니다.
# - 즉, 각 샘플은 image와 label의 튜플로 구성됩니다.
#
# with_info=True:
# - 데이터셋뿐만 아니라 데이터셋에 대한 정보도 함께 반환합니다.
# - 예를 들어 전체 샘플 개수, 클래스 이름, 클래스 개수 등을 확인할 수 있습니다.
dataset, info = tfds.load(
    "tf_flowers",
    as_supervised=True,
    with_info=True
)


# 전체 데이터셋의 샘플 개수를 확인합니다.
#
# info.splits["train"]:
# - tf_flowers 데이터셋은 기본적으로 train split만 가지고 있습니다.
#
# num_examples:
# - 해당 split에 들어 있는 샘플 개수입니다.
#
# tf_flowers의 전체 이미지 개수는 3670개입니다.
dataset_size = info.splits["train"].num_examples


# 데이터셋의 클래스 이름을 확인합니다.
#
# info.features["label"]:
# - label feature에 대한 정보를 가져옵니다.
#
# names:
# - 정수 레이블에 대응되는 실제 클래스 이름 리스트입니다.
#
# 예:
# label 0 → "dandelion"
# label 1 → "daisy"
# label 2 → "tulips"
# label 3 → "sunflowers"
# label 4 → "roses"
#
# 단, 클래스 순서는 TensorFlow Datasets 버전에 따라
# 확인해 보는 것이 좋습니다.
class_names = info.features["label"].names


# 클래스 개수를 확인합니다.
#
# tf_flowers는 5개의 꽃 클래스를 가지고 있습니다.
#
# 예:
# - dandelion
# - daisy
# - tulips
# - sunflowers
# - roses
n_classes = info.features["label"].num_classes


# 확인용 출력입니다.
#
# 예상 출력:
# 3670
# ['dandelion', 'daisy', ...]
# 5
print(dataset_size)
print(class_names)
print(n_classes)


# tf_flowers 데이터셋은 기본적으로 train split만 제공됩니다.
# 따라서 train split을 비율로 잘라서
# 테스트 세트, 검증 세트, 훈련 세트로 나눕니다.
#
# 여기서는 다음 비율로 나눕니다.
#
# 1. 처음 10%:
#    테스트 세트
#
# 2. 다음 15%:
#    검증 세트
#
# 3. 나머지 75%:
#    훈련 세트
#
# split 문법:
#
# "train[:10%]"
# - train 데이터의 처음 10%를 선택합니다.
#
# "train[10%:25%]"
# - train 데이터의 10% 지점부터 25% 지점까지 선택합니다.
# - 전체 기준으로 15%에 해당합니다.
#
# "train[25%:]"
# - train 데이터의 25% 지점 이후부터 끝까지 선택합니다.
# - 전체 기준으로 75%에 해당합니다.
test_set_raw, valid_set_raw, train_set_raw = tfds.load(
    "tf_flowers",
    split=[
        "train[:10%]",     # 전체 데이터의 처음 10%를 테스트 세트로 사용
        "train[10%:25%]",  # 그다음 15%를 검증 세트로 사용
        "train[25%:]"      # 나머지 75%를 훈련 세트로 사용
    ],
    as_supervised=True     # 각 샘플을 (이미지, 레이블) 형태로 반환
)


# 각 데이터셋의 샘플 개수를 확인합니다.
#
# tf.data.Dataset 객체는 바로 len()이 되지 않는 경우가 많으므로
# cardinality()를 사용합니다.
#
# 예상 개수는 대략 다음과 같습니다.
#
# 테스트 세트: 약 367개
# 검증 세트: 약 550개
# 훈련 세트: 약 2753개
print(tfds.as_dataframe(test_set_raw.take(1), info))  # 샘플 하나 확인용


# cardinality()는 데이터셋의 원소 개수를 Tensor 형태로 반환합니다.
#
# .numpy()를 붙이면 실제 정수값으로 확인할 수 있습니다.
print("테스트 세트 개수:", test_set_raw.cardinality().numpy())
print("검증 세트 개수:", valid_set_raw.cardinality().numpy())
print("훈련 세트 개수:", train_set_raw.cardinality().numpy())

In [ ]:
# 배치 크기를 32로 지정합니다.
#
# 배치(batch)란 모델이 한 번에 처리하는 샘플 묶음입니다.
#
# batch_size=32이면,
# 모델은 이미지를 1장씩 처리하는 것이 아니라
# 32장씩 묶어서 처리합니다.
batch_size = 32


# Xception 모델에 입력하기 전,
# 이미지 크기 변경과 전처리를 한 번에 수행하는 전처리 모델을 만듭니다.
#
# tf.keras.Sequential은 여러 층을 순서대로 연결하는 컨테이너입니다.
#
# 여기서는 다음 두 단계를 순서대로 수행합니다.
#
# 1. 이미지 크기를 224x224로 변경
# 2. Xception 모델이 기대하는 형식으로 픽셀값 전처리
preprocess = tf.keras.Sequential([

    # 이미지 크기를 224x224로 변경합니다.
    #
    # Xception 모델은 기본적으로 224x224 또는 그 이상의 RGB 이미지를 입력으로 받을 수 있습니다.
    # 여기서는 모든 이미지를 같은 크기인 224x224로 맞춥니다.
    #
    # crop_to_aspect_ratio=True:
    # - 원본 이미지의 가로세로 비율을 최대한 유지합니다.
    # - 이미지를 무작정 찌그러뜨리지 않고,
    #   필요한 부분을 중앙 기준으로 잘라낸 뒤 크기를 조정합니다.
    tf.keras.layers.Resizing(
        height=224,
        width=224,
        crop_to_aspect_ratio=True
    ),

    # Lambda 층은 사용자가 원하는 함수를 Keras 층처럼 사용할 수 있게 해 줍니다.
    #
    # tf.keras.applications.xception.preprocess_input:
    # - Xception 모델에 맞는 입력 전처리를 수행하는 함수입니다.
    #
    # Xception의 preprocess_input()은 픽셀값을 모델이 학습될 때 사용한 방식에 맞게 변환합니다.
    #
    # 일반적으로 원본 이미지 픽셀값은 0~255 범위입니다.
    # Xception의 preprocess_input()은 이를 대략 -1~1 범위로 변환합니다.
    #
    # 주의:
    # - 여기서는 따로 Rescaling(1 / 255)을 적용하지 않습니다.
    # - preprocess_input() 함수가 필요한 스케일 변환을 직접 처리합니다.
    tf.keras.layers.Lambda(
        tf.keras.applications.xception.preprocess_input
    )

])


# 원본 훈련 데이터셋 train_set_raw에 전처리를 적용합니다.
#
# train_set_raw의 각 원소는 다음 형태입니다.
#
# (X, y)
#
# X:
# - 꽃 이미지
#
# y:
# - 꽃 이미지의 정답 레이블
#
# map()은 데이터셋의 각 샘플에 함수를 적용합니다.
#
# lambda X, y: (preprocess(X), y)
#
# 의미:
# - 이미지 X에는 preprocess를 적용합니다.
# - 레이블 y는 그대로 둡니다.
#
# 주의:
# 사용자가 적은 코드에는 preprocess(x)처럼 소문자 x가 들어가 있었는데,
# lambda에서 받은 변수 이름은 대문자 X이므로 preprocess(X)로 써야 합니다.
train_set = train_set_raw.map(
    lambda X, y: (preprocess(X), y)
)


# 훈련 데이터셋을 섞고, 배치로 묶고, 프리페칭을 적용합니다.
train_set = train_set.shuffle(
    1000,     # 셔플 버퍼 크기입니다.
              #
              # 데이터셋에서 1000개 샘플을 버퍼에 담아 두고,
              # 그 안에서 무작위로 샘플을 뽑아 섞습니다.
              #
              # 버퍼 크기가 클수록 더 잘 섞이지만,
              # 메모리를 더 많이 사용합니다.

    seed=42   # 난수 시드입니다.
              #
              # 같은 seed를 사용하면 실행할 때마다
              # 비슷한 방식으로 데이터가 섞입니다.
              #
              # 실험 재현성을 위해 자주 사용합니다.
).batch(
    batch_size
              # 데이터를 batch_size개씩 묶습니다.
              #
              # 여기서는 batch_size=32이므로
              # 한 배치에 이미지 32장과 레이블 32개가 들어갑니다.
).prefetch(
    1
              # 프리페칭은 모델이 현재 배치를 학습하는 동안
              # 다음 배치를 미리 준비해 두는 기능입니다.
              #
              # 이를 통해 데이터 로딩 시간 때문에
              # GPU나 CPU가 쉬는 시간을 줄일 수 있습니다.
              #
              # prefetch(1)은 다음 배치 1개를 미리 준비한다는 뜻입니다.
)


# 검증 데이터셋에도 같은 전처리를 적용합니다.
#
# 검증 세트는 모델 훈련 중 성능을 확인하는 용도입니다.
#
# 검증 데이터는 훈련 데이터처럼 무작위로 섞을 필요가 없습니다.
# 따라서 shuffle()은 사용하지 않습니다.
valid_set = valid_set_raw.map(
    lambda X, y: (preprocess(X), y)
).batch(
    batch_size
)


# 테스트 데이터셋에도 같은 전처리를 적용합니다.
#
# 테스트 세트는 최종 모델 성능을 평가하는 용도입니다.
#
# 테스트 데이터도 순서를 섞을 필요가 없으므로
# shuffle()은 사용하지 않습니다.
test_set = test_set_raw.map(
    lambda X, y: (preprocess(X), y)
).batch(
    batch_size
)

In [ ]:
# 데이터 증식(data augmentation) 모델을 만듭니다.
#
# 데이터 증식은 훈련 이미지에 무작위 변형을 적용해
# 모델이 더 다양한 이미지를 본 것처럼 학습하게 만드는 기법입니다.
#
# 특히 훈련 데이터가 많지 않을 때 과대적합(overfitting)을 줄이는 데 도움이 됩니다.
#
# 여기서는 다음 3가지 변형을 사용합니다.
#
# 1. 수평 뒤집기
# 2. 약간 회전
# 3. 명암 대비 조절
data_augmentation = tf.keras.Sequential([

    # 이미지를 무작위로 좌우 반전합니다.
    #
    # mode="horizontal":
    # - 수평 방향, 즉 좌우 방향으로 뒤집습니다.
    #
    # 꽃 이미지는 좌우가 바뀌어도 여전히 같은 꽃이므로
    # 수평 뒤집기는 자연스러운 데이터 증식 방법입니다.
    #
    # seed=42:
    # - 난수 시드를 고정해 실험을 어느 정도 재현 가능하게 합니다.
    tf.keras.layers.RandomFlip(
        mode="horizontal",
        seed=42
    ),

    # 이미지를 무작위로 회전합니다.
    #
    # factor=0.05:
    # - 전체 한 바퀴의 5% 범위 안에서 회전한다는 뜻입니다.
    #
    # 한 바퀴는 360도이므로,
    # 360 * 0.05 = 18도입니다.
    #
    # 즉, 대략 -18도에서 +18도 사이로 이미지를 랜덤 회전합니다.
    #
    # 너무 크게 회전하면 이미지가 부자연스러워질 수 있으므로
    # 작은 값으로 설정합니다.
    tf.keras.layers.RandomRotation(
        factor=0.05,
        seed=42
    ),

    # 이미지의 대비를 무작위로 조절합니다.
    #
    # factor=0.2:
    # - 대비를 어느 정도 범위 안에서 랜덤하게 바꿉니다.
    #
    # 명암이나 조명 조건이 조금 달라져도
    # 같은 꽃으로 인식할 수 있도록 돕습니다.
    tf.keras.layers.RandomContrast(
        factor=0.2,
        seed=42
    )

])


# ImageNet 데이터셋에서 사전 훈련된 Xception 모델을 불러옵니다.
#
# Xception은 깊은 합성곱 신경망 구조 중 하나이며,
# 이미지 분류에서 좋은 성능을 내는 사전 훈련 모델입니다.
#
# weights="imagenet":
# - ImageNet 데이터셋으로 미리 학습된 가중치를 사용합니다.
#
# include_top=False:
# - Xception의 맨 위 분류기 부분을 제외합니다.
#
# 원래 Xception의 top 부분에는 보통 다음 층들이 포함됩니다.
#
# - GlobalAveragePooling2D
# - Dense(1000, activation="softmax")
#
# Dense(1000)은 ImageNet의 1000개 클래스를 분류하기 위한 층입니다.
#
# 하지만 우리는 tf_flowers의 5개 꽃 클래스를 분류해야 하므로,
# ImageNet용 출력층은 제거하고
# 우리 문제에 맞는 새로운 출력층을 붙입니다.
base_model = tf.keras.applications.xception.Xception(
    weights="imagenet",
    include_top=False
)


# base_model.output은 Xception의 합성곱 기반 부분이 출력하는 특성 맵입니다.
#
# include_top=False로 불러왔기 때문에,
# 출력은 아직 클래스 확률이 아니라 4차원 feature map입니다.
#
# 예를 들어 입력 이미지가 224x224x3이면
# base_model.output은 대략 다음과 같은 형태가 됩니다.
#
# (batch_size, 7, 7, 2048)
#
# 의미:
# - batch_size: 이미지 개수
# - 7: feature map의 높이
# - 7: feature map의 너비
# - 2048: feature map의 채널 수
#
# GlobalAveragePooling2D는 각 채널마다
# 7x7 공간 전체의 평균을 계산합니다.
#
# 결과:
# (batch_size, 7, 7, 2048)
# → (batch_size, 2048)
#
# 즉, 이미지 1장마다 2048개의 특징값을 가진 벡터로 바꿉니다.
avg = tf.keras.layers.GlobalAveragePooling2D()(
    base_model.output
)


# 새로운 출력층을 추가합니다.
#
# n_classes는 앞에서 데이터셋 정보에서 얻은 클래스 개수입니다.
#
# tf_flowers 데이터셋의 경우:
# n_classes = 5
#
# Dense(n_classes):
# - 각 클래스마다 하나의 출력 유닛을 둡니다.
#
# activation="softmax":
# - 각 클래스에 대한 확률을 출력합니다.
# - 출력 확률들의 합은 1입니다.
#
# 예:
# [0.05, 0.10, 0.70, 0.10, 0.05]
#
# 위와 같은 출력은 세 번째 클래스일 확률이 가장 높다는 뜻입니다.
output = tf.keras.layers.Dense(
    n_classes,
    activation="softmax"
)(avg)


# 최종 모델을 만듭니다.
#
# tf.keras.Model은 입력과 출력을 직접 지정해 모델을 만드는 함수형 API 방식입니다.
#
# inputs=base_model.input:
# - Xception 모델이 받는 입력 이미지를 그대로 최종 모델의 입력으로 사용합니다.
#
# outputs=output:
# - 우리가 새로 만든 꽃 분류 출력층을 최종 모델의 출력으로 사용합니다.
#
# 결과적으로 모델 구조는 다음과 같습니다.
#
# 입력 이미지
# → Xception 합성곱 기반 모델
# → GlobalAveragePooling2D
# → Dense(n_classes, softmax)
#
# 즉, ImageNet에서 학습한 Xception의 특징 추출 능력을 사용하고,
# 마지막 분류기만 tf_flowers 데이터셋에 맞게 새로 붙인 모델입니다.
model = tf.keras.Model(
    inputs=base_model.input,
    outputs=output
)


# 모델 구조를 확인합니다.
#
# Xception 기반 모델과 새로 추가한 전역 평균 풀링 층,
# Dense 출력층이 포함되어 있는지 확인할 수 있습니다.
model.summary()

In [ ]:
# 전이 학습(transfer learning)을 할 때는
# 훈련 초기에는 사전 훈련된 기반 모델(base_model)의 가중치를 동결하는 것이 좋습니다.
#
# 여기서 base_model은 ImageNet 데이터셋으로 미리 훈련된 Xception 모델입니다.
#
# 이 모델은 이미 일반적인 이미지 특징을 잘 추출하도록 학습되어 있습니다.
# 예를 들어:
# - 선
# - 모서리
# - 색상 패턴
# - 질감
# - 간단한 형태
# - 복잡한 물체의 일부 특징
#
# 같은 것들을 이미 어느 정도 알고 있습니다.
#
# 따라서 처음부터 모든 층을 다시 학습시키면
# 기존에 잘 학습된 가중치가 크게 망가질 수 있습니다.
#
# 그래서 훈련 초반에는 base_model의 가중치를 고정하고,
# 새로 추가한 출력층만 먼저 학습시킵니다.
for layer in base_model.layers:

    # trainable=False로 설정하면 해당 층의 가중치는 훈련 중 업데이트되지 않습니다.
    #
    # 즉, 역전파 과정에서 그래디언트가 계산되더라도
    # optimizer가 이 층의 가중치를 바꾸지 않습니다.
    #
    # 이 코드의 결과:
    # - Xception 기반 모델의 가중치: 동결
    # - 새로 추가한 Dense 출력층의 가중치: 학습됨
    layer.trainable = False


# SGD 옵티마이저를 생성합니다.
#
# SGD는 Stochastic Gradient Descent, 즉 확률적 경사 하강법입니다.
#
# learning_rate=0.1:
# - 학습률입니다.
# - 한 번의 업데이트에서 가중치를 얼마나 크게 바꿀지 정합니다.
# - 값이 너무 크면 학습이 불안정해질 수 있고,
#   값이 너무 작으면 학습이 너무 느려질 수 있습니다.
#
# momentum=0.9:
# - 모멘텀을 사용합니다.
# - 이전 업데이트 방향을 어느 정도 유지하여
#   학습이 더 빠르고 안정적으로 진행되도록 도와줍니다.
optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.1,
    momentum=0.9
)


# 모델을 컴파일합니다.
#
# compile()은 모델을 훈련하기 전에
# 손실 함수, 옵티마이저, 평가 지표를 설정하는 단계입니다.
model.compile(

    # sparse_categorical_crossentropy:
    # - 다중 클래스 분류 문제에서 사용하는 손실 함수입니다.
    # - 정답 레이블이 원-핫 인코딩이 아니라 정수 레이블일 때 사용합니다.
    #
    # 예:
    # daisy      → 0
    # dandelion  → 1
    # roses      → 2
    # sunflowers → 3
    # tulips     → 4
    #
    # 이런 식으로 y가 정수 하나로 주어질 때 사용합니다.
    #
    # 만약 정답이 [0, 0, 1, 0, 0]처럼 원-핫 벡터라면
    # categorical_crossentropy를 사용합니다.
    loss="sparse_categorical_crossentropy",

    # 위에서 만든 SGD 옵티마이저를 사용합니다.
    optimizer=optimizer,

    # 훈련 중 정확도를 함께 확인합니다.
    #
    # accuracy:
    # - 예측 클래스가 정답 클래스와 일치한 비율입니다.
    metrics=["accuracy"]
)


# 모델 훈련을 시작합니다.
#
# train_set:
# - 앞에서 만든 훈련 데이터셋입니다.
# - 이미지 전처리, shuffle, batch, prefetch가 적용되어 있습니다.
#
# validation_data=valid_set:
# - 각 epoch이 끝날 때마다 검증 세트 성능을 측정합니다.
# - 검증 성능은 모델이 훈련 데이터에만 과하게 맞춰지고 있는지 확인하는 데 사용됩니다.
#
# epochs=3:
# - 전체 훈련 데이터셋을 3번 반복해서 학습합니다.
#
# 현재 base_model의 층들은 동결되어 있으므로
# 이 단계에서는 새로 추가한 출력층 위주로 학습됩니다.
history = model.fit(
    train_set,
    validation_data=valid_set,
    epochs=3
)


# history 객체에는 훈련 과정에서 기록된 손실과 정확도 값이 저장됩니다.
#
# 예:
# history.history["loss"]
# history.history["accuracy"]
# history.history["val_loss"]
# history.history["val_accuracy"]
#
# 이를 이용하면 epoch별 훈련/검증 성능 변화를 확인할 수 있습니다.
history.history

In [ ]:
# 처음 몇 epoch 동안 훈련하면,
# 새로 추가한 분류기 부분은 어느 정도 잘 학습됩니다.
#
# 예를 들어 검증 정확도가 75~80% 정도까지 올라간 뒤
# 더 이상 크게 향상되지 않을 수 있습니다.
#
# 이는 새로 추가한 최상위 층,
# 즉 GlobalAveragePooling2D 뒤의 Dense 출력층이
# 어느 정도 잘 훈련되었다는 의미입니다.
#
# 이제는 사전 훈련된 Xception 기반 모델의 상위 층 일부를
# 동결 해제하여 미세 튜닝(fine-tuning)을 진행할 수 있습니다.
#
# 미세 튜닝이란:
# - ImageNet으로 미리 학습된 가중치를 완전히 새로 학습하는 것이 아니라,
# - 현재 데이터셋(tf_flowers)에 조금 더 잘 맞도록 일부 가중치를 조금씩 조정하는 과정입니다.


# base_model.layers[56:]은 base_model의 56번째 층부터 마지막 층까지를 의미합니다.
#
# 즉, Xception 기반 모델의 앞쪽 층들은 계속 동결하고,
# 뒤쪽 층들만 훈련 가능하게 만듭니다.
#
# 일반적으로 CNN의 앞쪽 층은 다음과 같은 일반적인 특징을 학습합니다.
#
# - 선
# - 모서리
# - 색상 변화
# - 간단한 질감
#
# 이런 특징은 대부분의 이미지 데이터셋에서 공통적으로 유용합니다.
#
# 반면 뒤쪽 층은 더 구체적인 고수준 특징을 학습합니다.
#
# 예:
# - 꽃잎 모양
# - 꽃 중심부 패턴
# - 특정 물체의 구조
#
# 따라서 전이 학습에서는 보통 앞쪽 층은 동결하고,
# 뒤쪽 층 일부만 동결 해제하여 새 데이터셋에 맞게 조정합니다.
for layer in base_model.layers[56:]:

    # trainable=True로 설정하면 해당 층의 가중치가
    # 훈련 중 업데이트될 수 있습니다.
    #
    # 이 코드의 결과:
    # - base_model의 0번째 층부터 55번째 층까지는 기존 설정대로 동결 상태
    # - base_model의 56번째 층부터 마지막 층까지는 훈련 가능 상태
    layer.trainable = True


# 층의 trainable 속성을 바꾼 뒤에는 반드시 모델을 다시 컴파일해야 합니다.
#
# 이유:
# - Keras는 compile() 시점에 어떤 가중치를 훈련할지 결정합니다.
# - 따라서 trainable=True 또는 False를 바꾼 뒤
#   compile()을 다시 하지 않으면 변경 사항이 제대로 반영되지 않을 수 있습니다.


# 미세 튜닝 단계에서는 학습률을 더 작게 설정하는 것이 일반적입니다.
#
# 앞 단계에서는 새 출력층만 학습했으므로 learning_rate=0.1을 사용했습니다.
#
# 하지만 이제는 사전 훈련된 Xception의 일부 가중치도 업데이트합니다.
# 이미 잘 학습된 가중치를 너무 크게 바꾸면 성능이 나빠질 수 있으므로,
# learning_rate=0.01처럼 더 작은 학습률을 사용합니다.
optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.9
)


# 모델을 다시 컴파일합니다.
#
# 손실 함수와 평가 지표는 이전과 같습니다.
#
# sparse_categorical_crossentropy:
# - 정답 레이블이 정수 형태인 다중 클래스 분류 문제에 사용합니다.
#
# metrics=["accuracy"]:
# - 훈련 및 검증 정확도를 확인합니다.
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)


# 미세 튜닝을 계속 진행합니다.
#
# train_set:
# - 훈련 데이터셋입니다.
#
# validation_data=valid_set:
# - 각 epoch 후 검증 성능을 측정합니다.
#
# epochs=10:
# - 전체 훈련 데이터셋을 10번 반복해서 학습합니다.
#
# 이 단계에서는:
# - 새로 추가한 Dense 출력층
# - base_model의 56번째 층 이후의 일부 상위 층
#
# 이 함께 학습됩니다.
#
# 앞쪽 층은 여전히 동결되어 있으므로,
# 일반적인 이미지 특징은 유지하면서
# 뒤쪽 고수준 특징만 꽃 데이터셋에 맞게 조정합니다.
history = model.fit(
    train_set,
    validation_data=valid_set,
    epochs=10
)


# 훈련 결과는 history 객체에 저장됩니다.
#
# 확인 가능한 주요 값:
#
# history.history["loss"]
# - 훈련 손실
#
# history.history["accuracy"]
# - 훈련 정확도
#
# history.history["val_loss"]
# - 검증 손실
#
# history.history["val_accuracy"]
# - 검증 정확도
#
# 미세 튜닝이 잘 되면 검증 정확도가
# 이전보다 더 향상될 수 있습니다.
history.history

In [ ]:
# 사진에서 물체의 종류를 분류하는 작업은 분류(classification) 문제입니다.
#
# 예:
# - 고양이
# - 강아지
# - 자동차
# - 꽃
#
# 반면 사진에서 물체가 어디에 있는지 예측하는 작업은
# 회귀(regression) 문제로 표현할 수 있습니다.
#
# 대표적인 방식은 물체를 감싸는 바운딩 박스(bounding box)를 예측하는 것입니다.
#
# 바운딩 박스는 보통 4개의 숫자로 표현합니다.
#
# 예:
# - 물체 중심의 x 좌표
# - 물체 중심의 y 좌표
# - 바운딩 박스의 너비
# - 바운딩 박스의 높이
#
# 즉, 모델이 이미지마다 4개의 연속적인 숫자를 예측하도록 만듭니다.


# ImageNet 데이터셋으로 사전 훈련된 Xception 모델을 불러옵니다.
#
# weights="imagenet":
# - ImageNet으로 미리 학습된 가중치를 사용합니다.
#
# include_top=False:
# - Xception의 최상위 분류기 부분을 제외합니다.
# - 원래 ImageNet 1000개 클래스를 분류하던 Dense 출력층을 제거합니다.
#
# 이렇게 하면 Xception을 특징 추출기(feature extractor)로 사용할 수 있습니다.
base_model = tf.keras.applications.xception.Xception(
    weights="imagenet",
    include_top=False
)


# base_model.output은 Xception의 합성곱 기반 부분이 출력하는 feature map입니다.
#
# include_top=False이므로 출력은 아직 클래스 확률이 아닙니다.
#
# 예를 들어 입력 이미지가 224x224x3이면,
# base_model.output은 대략 다음과 같은 4차원 텐서입니다.
#
# (batch_size, 7, 7, 2048)
#
# GlobalAveragePooling2D는 각 채널마다
# 7x7 공간 방향 전체의 평균을 계산합니다.
#
# 결과:
# (batch_size, 7, 7, 2048)
# → (batch_size, 2048)
#
# 이 벡터 avg는 이미지의 고수준 특징을 담고 있으며,
# 분류 출력층과 위치 출력층이 함께 사용합니다.
avg = tf.keras.layers.GlobalAveragePooling2D()(
    base_model.output
)


# 첫 번째 출력층: 클래스 분류 출력층입니다.
#
# n_classes:
# - 분류할 클래스 개수입니다.
# - 예를 들어 tf_flowers 데이터셋이면 n_classes=5입니다.
#
# activation="softmax":
# - 각 클래스에 대한 확률을 출력합니다.
# - 출력 확률들의 합은 1입니다.
#
# 이 출력은 물체가 어떤 클래스인지 예측합니다.
class_output = tf.keras.layers.Dense(
    n_classes,
    activation="softmax"
)(avg)


# 두 번째 출력층: 위치 회귀 출력층입니다.
#
# Dense(4):
# - 4개의 숫자를 출력합니다.
#
# 일반적으로 이 4개 값은 바운딩 박스를 나타냅니다.
#
# 예:
# - 중심 x 좌표
# - 중심 y 좌표
# - 너비
# - 높이
#
# 또는 데이터셋에 따라 다음처럼 표현할 수도 있습니다.
#
# - 왼쪽 위 x 좌표
# - 왼쪽 위 y 좌표
# - 오른쪽 아래 x 좌표
# - 오른쪽 아래 y 좌표
#
# 중요한 점:
# 학습 데이터의 정답 바운딩 박스 형식과
# 모델 출력 형식이 반드시 일치해야 합니다.
#
# activation을 지정하지 않았으므로 기본값은 선형 출력입니다.
# 즉, 실수 4개를 그대로 예측합니다.
#
# 회귀 문제에서는 보통 softmax나 sigmoid를 바로 쓰지 않고
# 선형 출력을 사용하는 경우가 많습니다.
#
# 단, 바운딩 박스 좌표를 0~1 사이로 정규화했다면
# sigmoid 활성화 함수를 사용하는 방식도 가능합니다.
loc_output = tf.keras.layers.Dense(4)(
    avg
)


# 최종 모델을 만듭니다.
#
# 이 모델은 입력은 하나지만 출력은 두 개입니다.
#
# 입력:
# - 이미지
#
# 출력:
# 1. class_output
#    - 클래스 확률
#
# 2. loc_output
#    - 바운딩 박스 좌표 4개
#
# 이런 구조를 다중 출력 모델(multi-output model)이라고 합니다.
model = tf.keras.Model(
    inputs=base_model.input,
    outputs=[
        class_output,
        loc_output
    ]
)


# 모델을 컴파일합니다.
#
# 출력이 두 개이므로 loss도 두 개를 지정합니다.
#
# 첫 번째 loss:
# sparse_categorical_crossentropy
# - class_output에 적용됩니다.
# - 정수 레이블을 사용하는 다중 클래스 분류 문제에 적합합니다.
#
# 두 번째 loss:
# mse
# - loc_output에 적용됩니다.
# - 바운딩 박스 좌표처럼 연속적인 숫자를 예측하는 회귀 문제에 적합합니다.
#
# loss_weights:
# - 두 손실의 중요도를 조절합니다.
#
# loss_weights=[0.8, 0.2]는 다음 의미입니다.
#
# 전체 손실 =
# 0.8 * 분류 손실 + 0.2 * 위치 회귀 손실
#
# 즉, 이 예시에서는 분류 손실을 더 중요하게 보고,
# 위치 예측 손실은 상대적으로 낮은 비중을 둡니다.
#
# 실제 문제에서는 데이터셋과 목표에 따라
# loss_weights 값을 조정해야 합니다.
model.compile(
    loss=[
        "sparse_categorical_crossentropy",
        "mse"
    ],
    loss_weights=[
        0.8,   # 분류 손실의 가중치
        0.2    # 위치 회귀 손실의 가중치
    ],
    optimizer=optimizer,

    # metrics=["accuracy"]를 지정하면 보통 첫 번째 출력인
    # class_output에 대한 정확도를 확인하는 용도로 사용됩니다.
    #
    # 위치 회귀 출력에는 accuracy가 적절하지 않습니다.
    # 바운딩 박스 예측 성능은 일반적으로 MSE, MAE, IoU 등을 사용해 평가합니다.
    metrics=["accuracy"]
)